<a href="https://colab.research.google.com/github/Decoding-Data-Science/airesidency/blob/main/Weather_LLM_wrapper_c11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install openai==0.28 requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.5/76.5 kB 2.7 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.54.0
    Uninstalling openai-2.54.0:
      Successfully uninstalled openai-2.54.0


In [16]:
import os
from google.colab import userdata

import openai
# Retrieve API keys from Colab's secure storage
openai_api_key = userdata.get("openai")

# Set them as environment variables

if openai_api_key:
    os.environ["OPENAI_API_KEY"] = openai_api_key

In [6]:
#openweather key =https://home.openweathermap.org/api_keys
import requests

def get_current_weather(location, unit='celsius'):
    weather_api_key = "a7b109315b6ced6a9cd5177bdb98ca82"
    base_url = f"http://api.openweathermap.org/data/2.5/weather?q={location}&appid={weather_api_key}&units=metric"
    response = requests.get(base_url)
    data = response.json()

    weather_description = data['weather'][0]['description']

    return {
        "location": location,
        "temperature": data['main']['temp'],
        "weather": weather_description
    }

### Raw JSON Output from OpenWeatherMap API

This cell demonstrates how to get and display the raw JSON response from the OpenWeatherMap API, which is the content of the `data` variable inside the `get_current_weather` function.

In [8]:
import json

weather_api_key = "a7b109315b6ced6a9cd5177bdb98ca82"
location = "London"
base_url = f"http://api.openweathermap.org/data/2.5/weather?q={location}&appid={weather_api_key}&units=metric"
response = requests.get(base_url)
raw_json_data = response.json()

print(json.dumps(raw_json_data, indent=2))

{
  "coord": {
    "lon": -0.1257,
    "lat": 51.5085
  },
  "weather": [
    {
      "id": 800,
      "main": "Clear",
      "description": "clear sky",
      "icon": "01d"
    }
  ],
  "base": "stations",
  "main": {
    "temp": 20.03,
    "feels_like": 19.34,
    "temp_min": 18.06,
    "temp_max": 21.16,
    "pressure": 1022,
    "humidity": 48,
    "sea_level": 1022,
    "grnd_level": 1018
  },
  "visibility": 10000,
  "wind": {
    "speed": 2.68,
    "deg": 117,
    "gust": 5.36
  },
  "clouds": {
    "all": 0
  },
  "dt": 1787396450,
  "sys": {
    "type": 2,
    "id": 2075535,
    "country": "GB",
    "sunrise": 1787374621,
    "sunset": 1787425816
  },
  "timezone": 3600,
  "id": 2643743,
  "name": "London",
  "cod": 200
}


In [9]:
print(get_current_weather("Dubai"))

{'location': 'Dubai', 'temperature': 42.96, 'weather': 'clear sky'}


In [11]:
functions = [
    {
        "name": "get_current_weather",
        "description": "Get the current weather in a given location",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "The city, e.g. San Francisco"
                },
                "unit": {
                    "type": "string",
                    "enum": ["celsius", "fahrenheit"]
                }
            },
            "required": ["location"]
        }
    }
]

In [12]:
functions

[{'name': 'get_current_weather',
  'description': 'Get the current weather in a given location',
  'parameters': {'type': 'object',
   'properties': {'location': {'type': 'string',
     'description': 'The city, e.g. San Francisco'},
    'unit': {'type': 'string', 'enum': ['celsius', 'fahrenheit']}},
   'required': ['location']}}]

In [14]:
#Function Definition and Initial Message Handling
import json
def weather_chat(user_message):
    messages=[]
    messages.append({"role": "user", "content": user_message})
    messages.append({"role": "assistant", "content": "You are a weather bot . Answer only in Celsius  answer only weather related information and politely decline if not related  "})



        # Sending Initial Message to OpenAI
    response = openai.ChatCompletion.create(
            model="gpt-3.5-turbo",
            temperature = 0.2,
            max_tokens=256,
            top_p=0.5,
            frequency_penalty=0,
            presence_penalty=0,
            messages=messages,
            functions=functions
        )

    #Handling Function Calls and Fetching Weather Data
    try:
        function_call = response['choices'][0]['message']['function_call']
        arguments = json.loads(function_call['arguments'])

        # Fetch weather data using the extracted arguments
        weather_data = get_current_weather(arguments['location'])

        # Append the function call and weather data to the messages
        messages.append({"role": "assistant", "content": None, "function_call": {"name": "get_current_weather", "arguments": str(arguments)}})
        messages.append({"role": "function", "name": "get_current_weather", "content": str(weather_data)})

#magic of llm
        response = openai.ChatCompletion.create(
            model="gpt-3.5-turbo",
            messages=messages,
            temperature = 0.2,
            max_tokens=256,
            top_p=0.5,
            frequency_penalty=0,
            presence_penalty=0
                   )

        return response['choices'][0]['message']['content']
    except Exception as e:
        return "I'm here to provide weather updates. Please ask me questions related to weather."



In [18]:
import os
from google.colab import userdata

import openai
# Retrieve API keys from Colab's secure storage
openai_api_key = userdata.get("openai")

# Set them as environment variables

if openai_api_key:
    os.environ["OPENAI_API_KEY"] = openai_api_key

In [22]:
import openai
openai.api_key = openai_api_key
weather_chat("what is weather of Sharjahee now")

'The current weather in Sharjah is 43.16°C with clear skies.'

In [21]:
#qc
weather_chat("كم درجة الحرارة في الشارقة؟ ")

'درجة الحرارة في الشارقة هي حالياً حوالي 43.16 درجة مئوية مع سماء صافية.'

In [23]:
!pip install gradio

In [24]:
# Define Gradio interface
import gradio as gr
iface = gr.Interface(
    fn=weather_chat,
    inputs=gr.Textbox(label="Weather Queries"),
    outputs=gr.Textbox(label="Weather Updates"),
    title = "Weather Bot",
    description = "Ask me anything about weather!"
)

# Launch the Gradio interface
iface.launch(share="True")

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6c02b742f41c3f2733.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
